# Notebook 1 — Setup

This notebook introduces the tooling and libraries for learning Pong from pixels.

The figures below are explanatory only — you do not need to run any code to see them.
Run only the code cells (imports and setup) when you want to try the APIs yourself.

![Mac project stack with uv, Gymnasium, NumPy, and PyTorch](../assets/stack-mac-uv.svg)

## Running this notebook

Locally, the environment comes from the repo — `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

On Colab, the kernel exists before the project does, so the cell below
clones the repo and installs what Colab lacks. Run it first.


In [ ]:
# Colab bootstrap. No-op when running locally.
import pathlib
import sys

IN_COLAB = "google.colab" in sys.modules

# False: install only what Colab lacks (seconds).
# True:  install the locked environment exactly, which is reproducible but
#        downloads ~1GB and replaces the torch build Colab ships with.
EXACT_ENV = False

if IN_COLAB:
    if not pathlib.Path("/content/pong-pytorch").exists():
        !git clone -q https://github.com/piyushahuja/pong-pytorch.git /content/pong-pytorch
    %cd /content/pong-pytorch

    if EXACT_ENV:
        !pip install -q uv
        !uv export --locked --no-dev --format requirements.txt -o /tmp/pong-req.txt
        !pip install -q -r /tmp/pong-req.txt
    else:
        !pip install -q gymnasium ale-py  # torch and matplotlib ship with Colab

    # Puts the pong package on the path without touching its dependencies.
    !pip install -q -e . --no-deps

    print("bootstrapped in", pathlib.Path.cwd())
else:
    print("local run: using the uv environment from uv.lock")


## Project setup

There is nothing to create: the repo *is* the project. From a clone,

```bash
uv sync --locked
```

provisions Python 3.12, installs the exact versions in `uv.lock`, and installs the
`pong` package itself in editable mode — which is what makes `import pong` work in
these notebooks without any `sys.path` juggling. Then use the project `.venv` as
your notebook kernel.

`--locked` fails rather than quietly re-resolving if `pyproject.toml` and `uv.lock`
have drifted, so everyone gets the same environment.

**Do not run `uv init` in a clone** — it would overwrite `pyproject.toml`. You only
need it when starting a project from nothing, and then it needs `--package`, not
`--no-package`, because the code lives in `src/pong/` and has to be importable:

```bash
uv init --package pong-pytorch
cd pong-pytorch
uv python pin 3.12
uv add torch gymnasium ale-py matplotlib pygame
uv add --dev ipykernel nbformat nbclient pytest
```

Adding a dependency later is the same `uv add`, and it changes two files —
`pyproject.toml` and `uv.lock`. Commit both together, or `uv sync --locked` breaks
for everyone else.


## Same policy API, different environments

A PyTorch policy talks to the world through Gymnasium. Under that API you can plug in MuJoCo (robots) or ALE (Atari Pong). The learning loop shape stays the same.

![PyTorch policy stacks for MuJoCo vs ALE Pong](../assets/stack-policy-envs.svg)

## ALE (`ale-py`)

ALE = Arcade Learning Environment. The `ale-py` package lets Gymnasium run Atari 2600 games like Pong.

It is an emulator-based research environment: it runs the actual game ROM, keeps internal state, accepts controller actions, and returns pixels and rewards.

ALE does not know about neural networks. It only simulates Atari:

$$
(\text{game state},\, \text{action})
\rightarrow
(\text{new state},\, \text{pixels},\, \text{reward})
$$

In [ ]:
import ale_py

## Gymnasium

Gymnasium is the RL API that connects an agent to an environment. The two calls you will use constantly:

```python
obs, info = env.reset()
obs, reward, terminated, truncated, info = env.step(action)
```

In [ ]:
import gymnasium as gym

## PyTorch (`torch`)

PyTorch is the deep-learning library for this project. You will use it to:

- define a policy network (`torch.nn`)
- run a forward pass from pixels to action probabilities
- compute a loss, call `.backward()`, and update weights with an optimizer

It does not simulate Pong; Gymnasium + ALE do that. Torch only owns the learnable policy and gradients.

In [ ]:
import torch
import torch.nn as nn

## Small utilities

These show up later for math helpers and saving checkpoints.

In [ ]:
import math  # e.g. sqrt
from pathlib import Path  # save checkpoints